# Visualize per-timestamp-labeled training inputs

Each model input is a single array `target` of shape `(F, 256 + 512 + 64)`:

```
[ normal_signal (256) | context (512) | future (64) ]
```

and a `future_labels` array of shape `(64,)` giving the 0/1 anomaly label of every future step.

This notebook loads the prepared pickle and plots a few samples with the three regions shaded and the anomalous future steps highlighted.

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

# Region lengths (must match inst_data_prepare_labeled.py)
NORMAL_SIGNAL_LENGTH = 256
CONTEXT_LENGTH = 512
PREDICTION_LENGTH = 64

PKL_PATH = "./prepared_data_labeled/train_model_inputs.pkl"

with open(PKL_PATH, "rb") as f:
    data = pickle.load(f)

print(f"Loaded {len(data)} model inputs from {PKL_PATH}")
print("Example target shape  :", data[0]["target"].shape)
print("Example labels shape  :", data[0]["future_labels"].shape)

In [ ]:
# Dataset-level statistics
anom_steps = sum(int(d["future_labels"].sum()) for d in data)
total_steps = sum(int(d["future_labels"].size) for d in data)
pairs_with_anom = sum(1 for d in data if d["future_labels"].any())
n_var = [d["target"].shape[0] for d in data]

print(f"Total samples              : {len(data)}")
print(f"Num variates (min/max)     : {min(n_var)} / {max(n_var)}")
print(f"Future steps total         : {total_steps}")
print(f"  normal                   : {total_steps - anom_steps} ({(total_steps - anom_steps)/total_steps*100:.1f}%)")
print(f"  anomaly                  : {anom_steps} ({anom_steps/total_steps*100:.1f}%)")
print(f"Samples with >=1 anomaly   : {pairs_with_anom} ({pairs_with_anom/len(data)*100:.1f}%)")

In [ ]:
def plot_sample(sample, max_variates=4, title=None):
    """Plot one model input: shade the three regions and mark anomalous future steps."""
    target = sample["target"]            # (F, 832)
    labels = sample["future_labels"]     # (64,)
    F, T = target.shape

    ns_end = NORMAL_SIGNAL_LENGTH
    ctx_end = NORMAL_SIGNAL_LENGTH + CONTEXT_LENGTH
    fut_start, fut_end = ctx_end, ctx_end + PREDICTION_LENGTH

    n = min(F, max_variates)
    fig, axes = plt.subplots(n, 1, figsize=(14, 2.4 * n), sharex=True)
    if n == 1:
        axes = [axes]

    x = np.arange(T)
    for v in range(n):
        ax = axes[v]
        ax.plot(x, target[v], lw=0.8, color="steelblue")

        # region shading
        ax.axvspan(0, ns_end, color="gold", alpha=0.12)
        ax.axvspan(ns_end, ctx_end, color="green", alpha=0.08)
        ax.axvspan(fut_start, fut_end, color="gray", alpha=0.10)
        ax.axvline(ns_end, color="k", ls=":", lw=0.7)
        ax.axvline(ctx_end, color="k", ls=":", lw=0.7)

        # highlight anomalous future steps
        anom_idx = np.nonzero(labels == 1)[0]
        for i in anom_idx:
            ax.axvspan(fut_start + i, fut_start + i + 1, color="red", alpha=0.35)

        ax.set_ylabel(f"var {v}")
        ax.margins(x=0)

    # region labels on top axis
    axes[0].text(ns_end / 2, 1.02, "normal_signal", transform=axes[0].get_xaxis_transform(),
                 ha="center", va="bottom", fontsize=9, color="darkgoldenrod")
    axes[0].text((ns_end + ctx_end) / 2, 1.02, "context", transform=axes[0].get_xaxis_transform(),
                 ha="center", va="bottom", fontsize=9, color="darkgreen")
    axes[0].text((fut_start + fut_end) / 2, 1.02, "future", transform=axes[0].get_xaxis_transform(),
                 ha="center", va="bottom", fontsize=9, color="dimgray")

    n_anom = int(labels.sum())
    fig.suptitle(title or f"sample — {n_anom}/{len(labels)} anomalous future steps", y=1.01)
    axes[-1].set_xlabel("time step")
    plt.tight_layout()
    plt.show()

## A sample whose future contains anomalies

In [ ]:
anom_samples = [d for d in data if d["future_labels"].any()]
if anom_samples:
    plot_sample(anom_samples[0])
else:
    print("No samples with anomalous future steps found.")

## A fully-normal sample

In [ ]:
normal_samples = [d for d in data if not d["future_labels"].any()]
if normal_samples:
    plot_sample(normal_samples[0])
else:
    print("No fully-normal samples found.")

## Zoom into the future window of an anomalous sample

Plots just the 64 future steps with each step coloured by its label (blue=normal, red=anomaly).

In [ ]:
def plot_future_zoom(sample, max_variates=4):
    target = sample["target"]
    labels = sample["future_labels"]
    fut = target[:, -PREDICTION_LENGTH:]
    F = fut.shape[0]
    n = min(F, max_variates)

    fig, axes = plt.subplots(n, 1, figsize=(12, 2.2 * n), sharex=True)
    if n == 1:
        axes = [axes]
    x = np.arange(PREDICTION_LENGTH)
    for v in range(n):
        ax = axes[v]
        ax.plot(x, fut[v], color="steelblue", lw=1.0, zorder=1)
        anom_mask = labels == 1
        ax.scatter(x[anom_mask], fut[v][anom_mask], color="red", s=18, zorder=3, label="anomaly")
        ax.scatter(x[~anom_mask], fut[v][~anom_mask], color="steelblue", s=12, zorder=2, label="normal")
        ax.set_ylabel(f"var {v}")
        ax.margins(x=0)
    axes[0].legend(loc="upper right", fontsize=8)
    axes[-1].set_xlabel("future step (0..63)")
    fig.suptitle("future window, coloured by per-step label", y=1.01)
    plt.tight_layout()
    plt.show()

if anom_samples:
    plot_future_zoom(anom_samples[0])